In [ ]:
!pip install -q transformers librosa soundfile datasets huggingface_hub torch torchaudio pandas scikit-learn tqdm onnx onnxruntime gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 25.9 MB/s eta 0:00:00


In [ ]:

import numpy as np, librosa, soundfile as sf, csv, time
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm
import os

TARGET_SR = 16000
out_dir = Path("data/processed")
audio_out = out_dir / "audio_whisper"
audio_out.mkdir(parents=True, exist_ok=True)

HINDI_QUOTA = 900
FILLER_QUOTA = 450
REST_QUOTA = 150
TOTAL_QUOTA = HINDI_QUOTA + FILLER_QUOTA + REST_QUOTA  # 1500 total

print("Streaming dataset (no full download)...")
ds_stream = load_dataset("pipecat-ai/smart-turn-data-v3.2-train", split="train", streaming=True)

rows = []
hindi_n = filler_n = rest_n = 0
t0 = time.time()

for ex in tqdm(ds_stream, desc="Streaming + filtering"):
    if len(rows) >= TOTAL_QUOTA:
        break
    is_hindi = ex["language"] == "hin"
    is_filler = bool(ex["midfiller"]) or bool(ex["endfiller"])

    take = False
    if is_hindi and hindi_n < HINDI_QUOTA:
        take = True; hindi_n += 1
    elif is_filler and not is_hindi and filler_n < FILLER_QUOTA:
        take = True; filler_n += 1
    elif not is_hindi and not is_filler and rest_n < REST_QUOTA:
        take = True; rest_n += 1
    if not take:
        continue

    try:
        audio = ex["audio"]
        wav = np.asarray(audio["array"], dtype=np.float32)
        sr = audio["sampling_rate"]
        if wav.size == 0 or not np.isfinite(wav).all():
            continue
        if sr != TARGET_SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
        out_path = audio_out / f"w_{len(rows):05d}.wav"
        sf.write(out_path, wav, TARGET_SR, subtype="PCM_16")
        rows.append({
            "audio_path": str(out_path), "label": int(ex["endpoint_bool"]),
            "language": ex["language"], "midfiller": bool(ex["midfiller"]),
            "endfiller": bool(ex["endfiller"]), "synthetic": bool(ex["synthetic"]),
        })
    except Exception:
        continue

print(f"\nDone in {(time.time()-t0)/60:.1f} min. Collected {len(rows)} "
      f"(Hindi={hindi_n}, filler={filler_n}, rest={rest_n})")

manifest_path = out_dir / "manifest_whisper.csv"
with open(manifest_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["audio_path", "label", "language", "midfiller", "endfiller", "synthetic"])
    writer.writeheader(); writer.writerows(rows)
print(f"Wrote {manifest_path}")

Streaming dataset (no full download)...


README.md:   0%|          | 0.00/3.26k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Streaming + filtering: 19059it [02:14, 142.00it/s]


Done in 2.2 min. Collected 1500 (Hindi=900, filler=450, rest=150)
Wrote data/processed/manifest_whisper.csv


In [ ]:

import torch
torch.set_num_threads(os.cpu_count())

from transformers import AutoFeatureExtractor, WhisperModel

MODEL_NAME = "openai/whisper-tiny"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
whisper_model = WhisperModel.from_pretrained(MODEL_NAME)
encoder = whisper_model.encoder
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

print("Encoder ready. Hidden size:", encoder.config.d_model)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Encoder ready. Hidden size: 384


In [ ]:

import pandas as pd, numpy as np

df = pd.read_csv("data/processed/manifest_whisper.csv")
print("Rows to process:", len(df))

TARGET_SR = 16000
BATCH_SIZE = 16
WINDOW_SECONDS = 2.0

def load_wav(path, target_len_s=WINDOW_SECONDS):
    wav, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    target_len = int(target_len_s * TARGET_SR)
    if len(wav) < target_len:
        wav = np.pad(wav, (target_len - len(wav), 0))
    else:
        wav = wav[-target_len:]
    return wav

paths = df["audio_path"].tolist()
labels = df["label"].tolist()

embeddings, kept_labels, failed = [], [], 0
t0 = time.time()

for i in tqdm(range(0, len(paths), BATCH_SIZE)):
    batch_paths = paths[i:i+BATCH_SIZE]
    batch_labels = labels[i:i+BATCH_SIZE]
    wavs, valid_labels = [], []
    for p, lab in zip(batch_paths, batch_labels):
        try:
            w = load_wav(p)
            if w.size == 0 or not np.isfinite(w).all():
                failed += 1; continue
            wavs.append(w); valid_labels.append(lab)
        except Exception:
            failed += 1
    if not wavs:
        continue
    inputs = feature_extractor(wavs, sampling_rate=TARGET_SR, return_tensors="pt")
    with torch.no_grad():
        out = encoder(inputs.input_features)
        pooled = out.last_hidden_state.mean(dim=1)
    embeddings.append(pooled)
    kept_labels.extend(valid_labels)

embeddings = torch.cat(embeddings, dim=0)
labels_t = torch.tensor(kept_labels, dtype=torch.long)

print(f"\nDone in {(time.time()-t0)/60:.1f} min. Failed: {failed}")
print("Embeddings shape:", embeddings.shape)
print("Label balance:", torch.bincount(labels_t).tolist())

torch.save({"embeddings": embeddings, "labels": labels_t}, "data/processed/whisper_embeddings_cache.pt")

Rows to process: 1500


100%|██████████| 94/94 [24:32<00:00, 15.67s/it]


Done in 24.5 min. Failed: 0
Embeddings shape: torch.Size([1500, 384])
Label balance: [807, 693]


In [ ]:

import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from torch.utils.data import TensorDataset, DataLoader, Subset

cache = torch.load("data/processed/whisper_embeddings_cache.pt")
X, y = cache["embeddings"], cache["labels"]
print("X:", X.shape, "y:", y.shape, "balance:", torch.bincount(y).tolist())

idx = np.arange(len(y))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=42, stratify=y.numpy())
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42, stratify=y[temp_idx].numpy())

full_ds = TensorDataset(X, y)
train_loader = DataLoader(Subset(full_ds, train_idx), batch_size=32, shuffle=True)
val_loader = DataLoader(Subset(full_ds, val_idx), batch_size=32)
test_loader = DataLoader(Subset(full_ds, test_idx), batch_size=32)


class WhisperHead(nn.Module):
    def __init__(self, in_dim=384, hidden=128, num_classes=2, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


head = WhisperHead(in_dim=X.shape[1])
print(f"Head params: {head.count_params():,}")

counts = np.bincount(y[train_idx].numpy(), minlength=2)
class_weights = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

@torch.no_grad()
def evaluate(loader):
    head.eval()
    preds_all, labels_all = [], []
    for xb, yb in loader:
        preds_all.append(head(xb).argmax(-1).numpy())
        labels_all.append(yb.numpy())
    preds_all, labels_all = np.concatenate(preds_all), np.concatenate(labels_all)
    p, r, f1, _ = precision_recall_fscore_support(labels_all, preds_all, labels=[1], average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels_all, preds_all), "precision_eot": p, "recall_eot": r, "f1_eot": f1}

best_f1, patience = -1.0, 0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(30):
    head.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(head(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    scheduler.step()
    vm = evaluate(val_loader)
    print(f"Epoch {epoch+1:02d} | loss={running_loss/len(train_idx):.4f} | "
          f"f1_eot={vm['f1_eot']:.4f} | precision={vm['precision_eot']:.4f} | recall={vm['recall_eot']:.4f}")
    if vm["f1_eot"] > best_f1:
        best_f1, patience = vm["f1_eot"], 0
        torch.save({"model_state": head.state_dict(), "in_dim": X.shape[1]}, "checkpoints/whisper_head_best.pt")
        print(f"  saved (f1={best_f1:.4f})")
    else:
        patience += 1
        if patience >= 6:
            print("Early stopping."); break

head.load_state_dict(torch.load("checkpoints/whisper_head_best.pt")["model_state"])
print("\nTest:", evaluate(test_loader))

X: torch.Size([1500, 384]) y: torch.Size([1500]) balance: [807, 693]
Head params: 49,538
Epoch 01 | loss=0.7043 | f1_eot=0.6480 | precision=0.5321 | recall=0.8286
  saved (f1=0.6480)
Epoch 02 | loss=0.6887 | f1_eot=0.6309 | precision=0.5949 | recall=0.6714
Epoch 03 | loss=0.6781 | f1_eot=0.6364 | precision=0.4667 | recall=1.0000
Epoch 04 | loss=0.6791 | f1_eot=0.0000 | precision=0.0000 | recall=0.0000
Epoch 05 | loss=0.6680 | f1_eot=0.6569 | precision=0.6716 | recall=0.6429
  saved (f1=0.6569)
Epoch 06 | loss=0.6587 | f1_eot=0.6980 | precision=0.6582 | recall=0.7429
  saved (f1=0.6980)
Epoch 07 | loss=0.6402 | f1_eot=0.6190 | precision=0.6964 | recall=0.5571
Epoch 08 | loss=0.6383 | f1_eot=0.7027 | precision=0.6667 | recall=0.7429
  saved (f1=0.7027)
Epoch 09 | loss=0.6233 | f1_eot=0.7005 | precision=0.5433 | recall=0.9857
Epoch 10 | loss=0.6186 | f1_eot=0.6569 | precision=0.6716 | recall=0.6429
Epoch 11 | loss=0.5962 | f1_eot=0.6950 | precision=0.6901 | recall=0.7000
Epoch 12 | loss=0

In [ ]:

test_df = df.iloc[test_idx].reset_index(drop=True)
head.eval()
with torch.no_grad():
    test_preds = head(X[test_idx]).argmax(-1).numpy()
test_df["pred"] = test_preds
test_df["true_label"] = y[test_idx].numpy()

def slice_report(name, sub):
    if len(sub) == 0:
        print(f"{name}: no examples"); return
    p, r, f1, _ = precision_recall_fscore_support(sub["true_label"], sub["pred"], labels=[1], average="binary", zero_division=0)
    acc = accuracy_score(sub["true_label"], sub["pred"])
    print(f"{name:25s} n={len(sub):5d}  acc={acc:.3f}  f1_eot={f1:.3f}  precision={p:.3f}  recall={r:.3f}")

print("Stratified test results (Whisper Tiny + head):")
slice_report("overall", test_df)
slice_report("language=hin", test_df[test_df["language"] == "hin"])
slice_report("midfiller=True", test_df[test_df["midfiller"] == True])
slice_report("endfiller=True", test_df[test_df["endfiller"] == True])
slice_report("synthetic=True", test_df[test_df["synthetic"] == True])
slice_report("synthetic=False", test_df[test_df["synthetic"] == False])

Stratified test results (Whisper Tiny + head):
overall                   n=  150  acc=0.753  f1_eot=0.773  precision=0.670  recall=0.913
language=hin              n=   86  acc=0.791  f1_eot=0.816  precision=0.702  recall=0.976
midfiller=True            n=   81  acc=0.765  f1_eot=0.796  precision=0.698  recall=0.925
endfiller=True            n=   51  acc=0.667  f1_eot=0.000  precision=0.000  recall=0.000
synthetic=True            n=  141  acc=0.752  f1_eot=0.777  precision=0.663  recall=0.938
synthetic=False           n=    9  acc=0.778  f1_eot=0.667  precision=1.000  recall=0.500


In [ ]:
!pip install -q onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 6.5 MB/s eta 0:00:00


In [ ]:

import torch, os
from onnxruntime.quantization import quantize_dynamic, QuantType

head.eval()
dummy_input = torch.randn(1, X.shape[1])

fp32_path = "checkpoints/whisper_head.onnx"
int8_path = "checkpoints/whisper_head_int8.onnx"

torch.onnx.export(
    head, dummy_input, fp32_path,
    input_names=["embedding"], output_names=["logits"],
    dynamic_axes={"embedding": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17, do_constant_folding=True, dynamo=False,
)
print(f"Exported fp32 -> {fp32_path}")

quantize_dynamic(model_input=fp32_path, model_output=int8_path, weight_type=QuantType.QInt8)
print(f"Exported int8 -> {int8_path}")

for label, path in [("head fp32", fp32_path), ("head int8", int8_path)]:
    print(f"{label:12s} {os.path.getsize(path)/1024:.2f} KB")

/tmp/ipykernel_582/3021955223.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported fp32 -> checkpoints/whisper_head.onnx
Exported int8 -> checkpoints/whisper_head_int8.onnx
head fp32    194.05 KB
head int8    51.17 KB


In [ ]:

import time
import numpy as np
import onnxruntime as ort

# Full pipeline latency: encoder forward pass + head forward pass, matching real inference
def full_pipeline_latency(runs=30):
    times = []
    sample_wav = np.random.randn(32000).astype(np.float32)  # 2s at 16kHz, random noise as stand-in
    for _ in range(5):  # warmup
        inputs = feature_extractor([sample_wav], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            pooled = encoder(inputs.input_features).last_hidden_state.mean(dim=1)
            _ = head(pooled)
    for _ in range(runs):
        t0 = time.perf_counter()
        inputs = feature_extractor([sample_wav], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            pooled = encoder(inputs.input_features).last_hidden_state.mean(dim=1)
            _ = head(pooled)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.percentile(times, 50), np.percentile(times, 95)

mean_ms, p50_ms, p95_ms = full_pipeline_latency()
print(f"Full pipeline (Whisper encoder + head), mean={mean_ms:.2f}ms  p50={p50_ms:.2f}ms  p95={p95_ms:.2f}ms")

# Rough total size: Whisper Tiny encoder weights + head
whisper_size_mb = sum(p.numel() * 4 for p in encoder.parameters()) / (1024**2)  # fp32, 4 bytes/param
head_size_kb = os.path.getsize(fp32_path) / 1024
print(f"Whisper Tiny encoder: ~{whisper_size_mb:.1f} MB")
print(f"Head: {head_size_kb:.2f} KB")
print(f"Total: ~{whisper_size_mb + head_size_kb/1024:.1f} MB")

Full pipeline (Whisper encoder + head), mean=1102.96ms  p50=780.35ms  p95=1907.97ms
Whisper Tiny encoder: ~31.3 MB
Head: 194.05 KB
Total: ~31.5 MB


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/turn-detection")
print("cwd:", os.getcwd())

Mounted at /content/drive
cwd: /content/drive/MyDrive/turn-detection


In [ ]:

%%writefile demo/app.py
import sys
from pathlib import Path

import gradio as gr

sys.path.append(str(Path(__file__).resolve().parent.parent))
from src.inference import TurnDetector

MODEL_PATH = "checkpoints/tiny_turn_net.onnx"
CONFIG_PATH = "configs/config.yaml"

detector = TurnDetector(MODEL_PATH, CONFIG_PATH)

LABELS = {False: "Still speaking / pause (don't interrupt)", True: "End of turn (safe to respond)"}


def predict(audio_path):
    if audio_path is None:
        return "No audio provided.", {}
    result = detector.predict_endpoint(audio_path)
    result_md = f"**{LABELS[result['is_end_of_turn']]}**\n\n_inference: {result['latency_ms']:.2f} ms_"
    confidence = {
        LABELS[True]: result["probs"]["end_of_turn"],
        LABELS[False]: result["probs"]["continuing"],
    }
    return result_md, confidence


with gr.Blocks(title="Tiny Turn Detection — Shiprocket Challenge") as demo:
    gr.Markdown(
        "# Tiny Turn Detection\n"
        "Record or upload the trailing ~2 seconds of a speaker's audio. "
        "TinyTurnNet (523K params, from-scratch CNN+GRU + prosodic features) predicts "
        "whether they've finished their turn or are pausing (including Hinglish fillers)."
    )
    audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Trailing audio window")
    run_btn = gr.Button("Predict", variant="primary")
    result_md = gr.Markdown()
    confidence_out = gr.Label(label="Class probabilities")
    run_btn.click(fn=predict, inputs=audio_in, outputs=[result_md, confidence_out])

if __name__ == "__main__":
    demo.launch()

Writing demo/app.py


In [ ]:

import os
print("demo/app.py:", os.path.exists("demo/app.py"))

demo/app.py: True


In [ ]:

import sys
for mod in list(sys.modules.keys()):
    if mod.startswith("demo") or mod.startswith("src"):
        del sys.modules[mod]

sys.path.insert(0, "/content/drive/MyDrive/turn-detection")
import gradio as gr
from src.inference import TurnDetector

MODEL_PATH = "checkpoints/tiny_turn_net.onnx"
CONFIG_PATH = "configs/config.yaml"
detector = TurnDetector(MODEL_PATH, CONFIG_PATH)
LABELS = {False: "Still speaking / pause (don't interrupt)", True: "End of turn (safe to respond)"}

def predict(audio_path):
    if audio_path is None:
        return "No audio provided.", {}
    result = detector.predict_endpoint(audio_path)
    result_md = f"**{LABELS[result['is_end_of_turn']]}**\n\n_inference: {result['latency_ms']:.2f} ms_"
    confidence = {LABELS[True]: result["probs"]["end_of_turn"], LABELS[False]: result["probs"]["continuing"]}
    return result_md, confidence

demo = gr.Interface(
    fn=predict,
    inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
    outputs=["markdown", gr.Label()],
    title="Tiny Turn Detection — Shiprocket Challenge",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://21280b3dbc60fb300a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

import torch, os
import torch.nn as nn

class WhisperHead(nn.Module):
    def __init__(self, in_dim=384, hidden=128, num_classes=2, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)

head = WhisperHead(in_dim=384)  # hardcoded from the known embedding size
head.load_state_dict(torch.load("whisper_head_best.pt")["model_state"])
head.eval()
print("Head loaded successfully")

os.makedirs("checkpoints", exist_ok=True)
import shutil
shutil.move("whisper_head_best.pt", "checkpoints/whisper_head_best.pt")

Head loaded successfully


'checkpoints/whisper_head_best.pt'

In [ ]:

from onnxruntime.quantization import quantize_dynamic, QuantType

dummy_input = torch.randn(1, 384)
fp32_path = "checkpoints/whisper_head.onnx"
int8_path = "checkpoints/whisper_head_int8.onnx"

torch.onnx.export(
    head, dummy_input, fp32_path,
    input_names=["embedding"], output_names=["logits"],
    dynamic_axes={"embedding": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17, do_constant_folding=True, dynamo=False,
)
quantize_dynamic(model_input=fp32_path, model_output=int8_path, weight_type=QuantType.QInt8)

for label, path in [("head fp32", fp32_path), ("head int8", int8_path)]:
    print(f"{label:12s} {os.path.getsize(path)/1024:.2f} KB")

/tmp/ipykernel_582/2053304591.py:7: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


head fp32    194.05 KB
head int8    51.17 KB


In [ ]:

import onnxruntime as ort
from sklearn.metrics import precision_recall_fscore_support

mels, prosodies, labels = cache["mels"], cache["prosodies"], cache["labels"]
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.2, random_state=cfg["train"]["seed"])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=cfg["train"]["seed"])

print("Test set size:", len(test_idx))

sess = ort.InferenceSession("checkpoints/tiny_turn_net.onnx", providers=["CPUExecutionProvider"])
input_names = [i.name for i in sess.get_inputs()]
output_name = sess.get_outputs()[0].name

test_mel = mels[test_idx].numpy().astype(np.float32)
test_prosody = prosodies[test_idx].numpy().astype(np.float32)
test_labels_arr = labels[test_idx].numpy()

logits = sess.run([output_name], {input_names[0]: test_mel, input_names[1]: test_prosody})[0]
exp_logits = np.exp(logits - logits.max(axis=-1, keepdims=True))
probs = exp_logits / exp_logits.sum(axis=-1, keepdims=True)
test_probs = probs[:, 1]  # P(end_of_turn)

print(f"{'Threshold':>10s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'FalsePos':>10s} {'FalseNeg':>10s}")
for thresh in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    preds = (test_probs >= thresh).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(test_labels_arr, preds, labels=[1], average="binary", zero_division=0)
    false_pos = int(((preds == 1) & (test_labels_arr == 0)).sum())
    false_neg = int(((preds == 0) & (test_labels_arr == 1)).sum())
    print(f"{thresh:10.2f} {p:10.3f} {r:10.3f} {f1:10.3f} {false_pos:10d} {false_neg:10d}")

Test set size: 500
 Threshold  Precision     Recall         F1   FalsePos   FalseNeg
      0.40      0.708      0.909      0.796         87         21
      0.45      0.708      0.879      0.785         84         28
      0.50      0.719      0.871      0.788         79         30
      0.55      0.733      0.853      0.789         72         34
      0.60      0.744      0.841      0.789         67         37
      0.65      0.743      0.823      0.781         66         41
      0.70      0.759      0.815      0.786         60         43
      0.75      0.766      0.789      0.777         56         49
      0.80      0.770      0.763      0.766         53         55


In [ ]:

import os
import numpy as np
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType, QuantFormat

# Calibration data reader using real cached features as calibration samples
class TurnNetCalibrationReader(CalibrationDataReader):
    def __init__(self, mels, prosodies, n_samples=200):
        self.mels = mels[:n_samples].numpy().astype(np.float32)
        self.prosodies = prosodies[:n_samples].numpy().astype(np.float32)
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.mels):
            return None
        item = {
            "mel": self.mels[self.idx:self.idx+1],
            "prosody": self.prosodies[self.idx:self.idx+1],
        }
        self.idx += 1
        return item

    def rewind(self):
        self.idx = 0

cache = torch.load("data/processed/features_cache.pt")
calib_reader = TurnNetCalibrationReader(cache["mels"], cache["prosodies"], n_samples=200)

fp32_path = "checkpoints/tiny_turn_net.onnx"
static_int8_path = "checkpoints/tiny_turn_net_static_int8.onnx"

quantize_static(
    model_input=fp32_path,
    model_output=static_int8_path,
    calibration_data_reader=calib_reader,
    quant_format=QuantFormat.QDQ,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QInt8,
)

print(f"Static int8 exported -> {static_int8_path}")
print(f"Size: {os.path.getsize(static_int8_path)/1024:.2f} KB")

Static int8 exported -> checkpoints/tiny_turn_net_static_int8.onnx
Size: 1992.95 KB


In [ ]:

import time

def bench(path, runs=100, warmup=10):
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    mel = np.random.randn(1, 1, cfg["audio"]["n_mels"], mels.shape[-1]).astype(np.float32)
    prosody = np.random.randn(1, cfg["model"]["prosody_feat_dim"]).astype(np.float32)
    feed = {"mel": mel, "prosody": prosody}
    for _ in range(warmup):
        sess.run(["logits"], feed)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        sess.run(["logits"], feed)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.percentile(times, 95)

print(f"{'Model':20s} {'Size(KB)':>10s} {'Mean(ms)':>10s} {'p95(ms)':>10s}")
for label, path in [
    ("fp32", "checkpoints/tiny_turn_net.onnx"),
    ("dynamic int8", "checkpoints/tiny_turn_net_int8.onnx"),
    ("static int8", "checkpoints/tiny_turn_net_static_int8.onnx"),
]:
    mean_ms, p95_ms = bench(path)
    size_kb = os.path.getsize(path) / 1024
    print(f"{label:20s} {size_kb:10.1f} {mean_ms:10.2f} {p95_ms:10.2f}")

Model                  Size(KB)   Mean(ms)    p95(ms)
fp32                     2049.5       6.60      17.33
dynamic int8             1990.3      27.22      52.41
static int8              1992.9       6.96      15.99


In [ ]:

sess_static = ort.InferenceSession(static_int8_path, providers=["CPUExecutionProvider"])
logits_static = sess_static.run(["logits"], {"mel": test_mel, "prosody": test_prosody})[0]
exp_l = np.exp(logits_static - logits_static.max(axis=-1, keepdims=True))
probs_static = exp_l / exp_l.sum(axis=-1, keepdims=True)
preds_static = (probs_static[:, 1] >= 0.5).astype(int)

from sklearn.metrics import precision_recall_fscore_support, accuracy_score
p, r, f1, _ = precision_recall_fscore_support(test_labels_arr, preds_static, labels=[1], average="binary", zero_division=0)
acc = accuracy_score(test_labels_arr, preds_static)
print(f"Static int8 test accuracy: {acc:.3f}, F1: {f1:.3f}, precision: {p:.3f}, recall: {r:.3f}")

Static int8 test accuracy: 0.778, F1: 0.783, precision: 0.717, recall: 0.862


In [ ]:

%%writefile configs/config_8s.yaml
audio:
  sample_rate: 16000
  window_seconds: 8.0
  n_mels: 64
  n_fft: 400
  hop_length: 160
  fmin: 20
  fmax: 8000

model:
  conv_channels: [32, 64, 96]
  gru_hidden: 96
  gru_layers: 1
  bidirectional: true
  prosody_feat_dim: 6
  dropout: 0.25
  num_classes: 2

train:
  batch_size: 64
  epochs: 25
  lr: 3.0e-4
  weight_decay: 1.0e-4
  label_smoothing: 0.05
  early_stopping_patience: 5
  seed: 42

Writing configs/config_8s.yaml


In [ ]:

import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import torch, time, yaml, pandas as pd

with open("configs/config_8s.yaml") as f:
    cfg_8s = yaml.safe_load(f)

from src.extract_worker import extract_one

df_8s = pd.read_csv("data/processed/manifest_tiny.csv")
print("Rows to process:", len(df_8s))
tasks = [(row["audio_path"], int(row["label"]), cfg_8s["audio"]) for _, row in df_8s.iterrows()]

N_WORKERS = min(mp.cpu_count(), 8)
print(f"Using {N_WORKERS} workers for {len(tasks)} clips at 8s window")

mels, prosodies, labels = [], [], []
failed = 0

t0 = time.time()
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    for mel, prosody, label, err in tqdm(ex.map(extract_one, tasks, chunksize=16), total=len(tasks)):
        if err is not None:
            failed += 1
            continue
        mels.append(torch.from_numpy(mel))
        prosodies.append(torch.from_numpy(prosody))
        labels.append(label)

print(f"Done in {(time.time()-t0)/60:.1f} min. Failed: {failed}")

max_t = max(m.shape[-1] for m in mels)
mels_padded = torch.stack([
    torch.nn.functional.pad(m, (0, max_t - m.shape[-1])) if m.shape[-1] < max_t else m
    for m in mels
]).unsqueeze(1)
prosodies_t = torch.stack(prosodies)
labels_t = torch.tensor(labels, dtype=torch.long)

cache_path_8s = "data/processed/features_cache_8s.pt"
torch.save({"mels": mels_padded, "prosodies": prosodies_t, "labels": labels_t}, cache_path_8s)
print(f"Cached {len(labels)} examples -> {cache_path_8s}")
print("mels shape:", mels_padded.shape)

Rows to process: 5000
Using 2 workers for 5000 clips at 8s window


100%|██████████| 5000/5000 [38:28<00:00,  2.17it/s]


Done in 38.5 min. Failed: 0
Cached 5000 examples -> data/processed/features_cache_8s.pt
mels shape: torch.Size([5000, 1, 64, 801])


In [ ]:

from google.colab import files
files.download("data/processed/features_cache_8s.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os, random, time
import numpy as np, torch, torch.nn as nn
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

from src.cached_dataset import CachedTurnDataset
from src.model import TinyTurnNet

torch.set_num_threads(os.cpu_count())

random.seed(cfg_8s["train"]["seed"]); np.random.seed(cfg_8s["train"]["seed"]); torch.manual_seed(cfg_8s["train"]["seed"])

full_ds_8s = CachedTurnDataset("data/processed/features_cache_8s.pt")
print("Dataset size:", len(full_ds_8s))

idx = np.arange(len(full_ds_8s))
train_idx_8s, temp_idx_8s = train_test_split(idx, test_size=0.2, random_state=cfg_8s["train"]["seed"])
val_idx_8s, test_idx_8s = train_test_split(temp_idx_8s, test_size=0.5, random_state=cfg_8s["train"]["seed"])

train_loader_8s = DataLoader(Subset(full_ds_8s, train_idx_8s), batch_size=cfg_8s["train"]["batch_size"], shuffle=True, drop_last=True)
val_loader_8s = DataLoader(Subset(full_ds_8s, val_idx_8s), batch_size=cfg_8s["train"]["batch_size"], shuffle=False)
test_loader_8s = DataLoader(Subset(full_ds_8s, test_idx_8s), batch_size=cfg_8s["train"]["batch_size"], shuffle=False)

model_8s = TinyTurnNet(
    n_mels=cfg_8s["audio"]["n_mels"], conv_channels=cfg_8s["model"]["conv_channels"],
    gru_hidden=cfg_8s["model"]["gru_hidden"], gru_layers=cfg_8s["model"]["gru_layers"],
    bidirectional=cfg_8s["model"]["bidirectional"], prosody_feat_dim=cfg_8s["model"]["prosody_feat_dim"],
    num_classes=cfg_8s["model"]["num_classes"], dropout=cfg_8s["model"]["dropout"],
)
print(f"Params: {model_8s.count_params():,}")

labels_arr = full_ds_8s.labels[train_idx_8s].numpy()
counts = np.bincount(labels_arr, minlength=2)
class_weights = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=cfg_8s["train"]["label_smoothing"])
optimizer = torch.optim.AdamW(model_8s.parameters(), lr=cfg_8s["train"]["lr"], weight_decay=cfg_8s["train"]["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg_8s["train"]["epochs"])

@torch.no_grad()
def evaluate_8s(loader):
    model_8s.eval()
    preds_all, labels_all = [], []
    for mel, prosody, label in loader:
        preds_all.append(model_8s(mel, prosody).argmax(-1).numpy())
        labels_all.append(label.numpy())
    preds_all, labels_all = np.concatenate(preds_all), np.concatenate(labels_all)
    p, r, f1, _ = precision_recall_fscore_support(labels_all, preds_all, labels=[1], average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels_all, preds_all), "precision_eot": p, "recall_eot": r, "f1_eot": f1}

best_f1, patience = -1.0, 0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(cfg_8s["train"]["epochs"]):
    model_8s.train()
    running_loss = 0.0
    t0 = time.time()
    for mel, prosody, label in tqdm(train_loader_8s, desc=f"Epoch {epoch+1}", leave=False):
        optimizer.zero_grad()
        loss = criterion(model_8s(mel, prosody), label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_8s.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item() * mel.size(0)
    scheduler.step()
    epoch_time = time.time() - t0

    vm = evaluate_8s(val_loader_8s)
    print(f"Epoch {epoch+1:02d} | {epoch_time:.1f}s | loss={running_loss/len(train_idx_8s):.4f} | "
          f"f1_eot={vm['f1_eot']:.4f} | precision={vm['precision_eot']:.4f} | recall={vm['recall_eot']:.4f}")

    if vm["f1_eot"] > best_f1:
        best_f1, patience = vm["f1_eot"], 0
        torch.save({"model_state": model_8s.state_dict(), "config": cfg_8s}, "checkpoints/tiny_turn_net_8s_best.pt")
        print(f"  saved (f1={best_f1:.4f})")
    else:
        patience += 1
        if patience >= cfg_8s["train"]["early_stopping_patience"]:
            print("Early stopping.")
            break

model_8s.load_state_dict(torch.load("checkpoints/tiny_turn_net_8s_best.pt", map_location="cpu")["model_state"])
print("\nTest (8s window):", evaluate_8s(test_loader_8s))

Dataset size: 5000
Params: 523,298


Epoch 01 | 259.1s | loss=0.7567 | f1_eot=0.5654 | precision=0.4807 | recall=0.6864
  saved (f1=0.5654)


Epoch 02 | 296.6s | loss=0.6941 | f1_eot=0.6384 | precision=0.4965 | recall=0.8941
  saved (f1=0.6384)


Epoch 03 | 243.6s | loss=0.6549 | f1_eot=0.6882 | precision=0.5755 | recall=0.8559
  saved (f1=0.6882)


Epoch 04 | 267.9s | loss=0.5524 | f1_eot=0.7709 | precision=0.6877 | recall=0.8771
  saved (f1=0.7709)


...

Epoch 07 | 217.5s | loss=0.3934 | f1_eot=0.7975 | precision=0.7941 | recall=0.8008


Epoch 08 | 217.5s | loss=0.3544 | f1_eot=0.8008 | precision=0.7695 | recall=0.8347


Epoch 09 | 230.9s | loss=0.3087 | f1_eot=0.7789 | precision=0.7741 | recall=0.7839


Epoch 10 | 221.0s | loss=0.2615 | f1_eot=0.7780 | precision=0.7490 | recall=0.8093


Epoch 11 | 237.9s | loss=0.2257 | f1_eot=0.7430 | precision=0.7577 | recall=0.7288
Early stopping.

Test (8s window): {'accuracy': 0.81, 'precision_eot': 0.7509157509157509, 'recall_eot': 0.8836206896551724, 'f1_eot': 0.8118811881188119}


In [ ]:

import torch, yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.cached_dataset import CachedTurnDataset
from src.model import TinyTurnNet

with open("configs/config_8s.yaml") as f:
    cfg_8s = yaml.safe_load(f)

df_8s = pd.read_csv("data/processed/manifest_tiny.csv")

full_ds_8s = CachedTurnDataset("data/processed/features_cache_8s.pt")
print("Dataset size:", len(full_ds_8s))

idx = np.arange(len(full_ds_8s))
train_idx_8s, temp_idx_8s = train_test_split(idx, test_size=0.2, random_state=cfg_8s["train"]["seed"])
val_idx_8s, test_idx_8s = train_test_split(temp_idx_8s, test_size=0.5, random_state=cfg_8s["train"]["seed"])

model_8s = TinyTurnNet(
    n_mels=cfg_8s["audio"]["n_mels"], conv_channels=cfg_8s["model"]["conv_channels"],
    gru_hidden=cfg_8s["model"]["gru_hidden"], gru_layers=cfg_8s["model"]["gru_layers"],
    bidirectional=cfg_8s["model"]["bidirectional"], prosody_feat_dim=cfg_8s["model"]["prosody_feat_dim"],
    num_classes=cfg_8s["model"]["num_classes"], dropout=cfg_8s["model"]["dropout"],
)
model_8s.load_state_dict(torch.load("checkpoints/tiny_turn_net_8s_best.pt", map_location="cpu")["model_state"])
model_8s.eval()
print("Model loaded and ready")

Dataset size: 5000
Model loaded and ready


In [ ]:

from sklearn.metrics import precision_recall_fscore_support, accuracy_score

test_df_8s = df_8s.iloc[test_idx_8s].reset_index(drop=True)
with torch.no_grad():
    mels_test_8s = full_ds_8s.mels[test_idx_8s]
    prosody_test_8s = full_ds_8s.prosodies[test_idx_8s]
    test_preds_8s = model_8s(mels_test_8s, prosody_test_8s).argmax(-1).numpy()
test_df_8s["pred"] = test_preds_8s
test_df_8s["true_label"] = full_ds_8s.labels[test_idx_8s].numpy()

def slice_report_8s(name, sub):
    if len(sub) == 0:
        print(f"{name}: no examples"); return
    p, r, f1, _ = precision_recall_fscore_support(sub["true_label"], sub["pred"], labels=[1], average="binary", zero_division=0)
    acc = accuracy_score(sub["true_label"], sub["pred"])
    print(f"{name:25s} n={len(sub):5d}  acc={acc:.3f}  f1_eot={f1:.3f}  precision={p:.3f}  recall={r:.3f}")

print("Stratified test results (8s window):")
slice_report_8s("overall", test_df_8s)
slice_report_8s("language=hin", test_df_8s[test_df_8s["language"] == "hin"])
slice_report_8s("midfiller=True", test_df_8s[test_df_8s["midfiller"] == True])
slice_report_8s("endfiller=True", test_df_8s[test_df_8s["endfiller"] == True])
slice_report_8s("synthetic=True", test_df_8s[test_df_8s["synthetic"] == True])
slice_report_8s("synthetic=False", test_df_8s[test_df_8s["synthetic"] == False])

Stratified test results (8s window):
overall                   n=  500  acc=0.810  f1_eot=0.812  precision=0.751  recall=0.884
language=hin              n=  296  acc=0.821  f1_eot=0.837  precision=0.739  recall=0.965
midfiller=True            n=  296  acc=0.760  f1_eot=0.784  precision=0.701  recall=0.890
endfiller=True            n=  166  acc=0.916  f1_eot=0.000  precision=0.000  recall=0.000
synthetic=True            n=  476  acc=0.824  f1_eot=0.826  precision=0.765  recall=0.896
synthetic=False           n=   24  acc=0.542  f1_eot=0.522  precision=0.462  recall=0.600


In [ ]:

import torch, os
torch.set_num_threads(os.cpu_count())
from transformers import AutoFeatureExtractor, WhisperModel

MODEL_NAME = "openai/whisper-tiny"
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
whisper_model = WhisperModel.from_pretrained(MODEL_NAME)
encoder = whisper_model.encoder
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False
print("Encoder reloaded. Hidden size:", encoder.config.d_model)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  151MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Encoder reloaded. Hidden size: 384


In [ ]:

import time

test_batch = df_full.iloc[:64]["audio_path"].tolist()
wavs_test = [load_wav(p) for p in test_batch]

times = []
for i in range(0, len(wavs_test), 16):
    batch = wavs_test[i:i+16]
    t0 = time.time()
    inputs = feature_extractor(batch, sampling_rate=TARGET_SR, return_tensors="pt")
    with torch.no_grad():
        out = encoder(inputs.input_features)
        pooled = out.last_hidden_state.mean(dim=1)
    times.append(time.time() - t0)
    print(f"Batch {i//16 + 1}: {times[-1]:.2f}s")

steady_state = sum(times[1:]) / max(len(times) - 1, 1)
print(f"\nSteady-state per batch: {steady_state:.2f}s")
print(f"Estimated full run (5000 rows, batch 16): {steady_state * (5000/16) / 60:.1f} min")

Batch 1: 29.32s
Batch 2: 29.25s
Batch 3: 27.69s
Batch 4: 14.49s

Steady-state per batch: 23.81s
Estimated full run (5000 rows, batch 16): 124.0 min


In [ ]:

import os
import torch
import numpy as np

CHUNK_SIZE = 625  # 8 chunks total
chunk_dir = "data/processed/whisper_chunks"
os.makedirs(chunk_dir, exist_ok=True)

n_chunks = (len(df_full) + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Total clips: {len(df_full)}, chunk size: {CHUNK_SIZE}, chunks: {n_chunks}")

for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(chunk_dir, f"chunk_{chunk_idx:02d}.pt")
    if os.path.exists(chunk_path):
        print(f"Chunk {chunk_idx} already done, skipping")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, len(df_full))
    chunk_df = df_full.iloc[start:end]

    paths = chunk_df["audio_path"].tolist()
    labels = chunk_df["label"].tolist()

    embeddings, kept_labels, failed = [], [], 0
    t0 = time.time()

    for i in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[i:i+BATCH_SIZE]
        batch_labels = labels[i:i+BATCH_SIZE]
        wavs, valid_labels = [], []
        for p, lab in zip(batch_paths, batch_labels):
            try:
                w = load_wav(p)
                if w.size == 0 or not np.isfinite(w).all():
                    failed += 1; continue
                wavs.append(w); valid_labels.append(lab)
            except Exception:
                failed += 1
        if not wavs:
            continue
        inputs = feature_extractor(wavs, sampling_rate=TARGET_SR, return_tensors="pt")
        with torch.no_grad():
            out = encoder(inputs.input_features)
            pooled = out.last_hidden_state.mean(dim=1)
        embeddings.append(pooled)
        kept_labels.extend(valid_labels)

    if embeddings:
        chunk_embeddings = torch.cat(embeddings, dim=0)
        chunk_labels_t = torch.tensor(kept_labels, dtype=torch.long)
        torch.save({"embeddings": chunk_embeddings, "labels": chunk_labels_t}, chunk_path)
        elapsed = time.time() - t0
        print(f"Chunk {chunk_idx} done: {len(kept_labels)} clips, {elapsed/60:.1f} min, {failed} failed -> {chunk_path}")
    else:
        print(f"Chunk {chunk_idx}: no valid clips!")

print("\nAll chunks complete.")

Total clips: 5000, chunk size: 625, chunks: 8
Chunk 0 already done, skipping
Chunk 1 done: 625 clips, 11.3 min, 0 failed -> data/processed/whisper_chunks/chunk_01.pt
Chunk 2 done: 625 clips, 17.1 min, 0 failed -> data/processed/whisper_chunks/chunk_02.pt
Chunk 3 done: 625 clips, 17.5 min, 0 failed -> data/processed/whisper_chunks/chunk_03.pt
Chunk 4 done: 625 clips, 17.8 min, 0 failed -> data/processed/whisper_chunks/chunk_04.pt
Chunk 5 done: 625 clips, 17.6 min, 0 failed -> data/processed/whisper_chunks/chunk_05.pt
Chunk 6 done: 625 clips, 17.5 min, 0 failed -> data/processed/whisper_chunks/chunk_06.pt
Chunk 7 done: 625 clips, 17.7 min, 0 failed -> data/processed/whisper_chunks/chunk_07.pt

All chunks complete.


In [ ]:

import glob

chunk_files = sorted(glob.glob(os.path.join(chunk_dir, "chunk_*.pt")))
print(f"Found {len(chunk_files)} chunks")

all_embeddings, all_labels = [], []
for cf in chunk_files:
    data = torch.load(cf)
    all_embeddings.append(data["embeddings"])
    all_labels.append(data["labels"])

full_embeddings = torch.cat(all_embeddings, dim=0)
full_labels = torch.cat(all_labels, dim=0)

print("Combined shape:", full_embeddings.shape)
print("Label balance:", torch.bincount(full_labels).tolist())

torch.save({"embeddings": full_embeddings, "labels": full_labels}, "data/processed/whisper_embeddings_cache_full.pt")
print("Saved combined cache")

Found 8 chunks
Combined shape: torch.Size([5000, 384])
Label balance: [2642, 2358]
Saved combined cache


In [ ]:

import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from torch.utils.data import TensorDataset, DataLoader, Subset

cache_full = torch.load("data/processed/whisper_embeddings_cache_full.pt")
X_full, y_full = cache_full["embeddings"], cache_full["labels"]
print("X:", X_full.shape, "y:", y_full.shape, "balance:", torch.bincount(y_full).tolist())

idx = np.arange(len(y_full))
train_idx_full, temp_idx_full = train_test_split(idx, test_size=0.2, random_state=42, stratify=y_full.numpy())
val_idx_full, test_idx_full = train_test_split(temp_idx_full, test_size=0.5, random_state=42, stratify=y_full[temp_idx_full].numpy())

full_ds_whisper = TensorDataset(X_full, y_full)
train_loader_full = DataLoader(Subset(full_ds_whisper, train_idx_full), batch_size=64, shuffle=True)
val_loader_full = DataLoader(Subset(full_ds_whisper, val_idx_full), batch_size=64)
test_loader_full = DataLoader(Subset(full_ds_whisper, test_idx_full), batch_size=64)


class WhisperHead(nn.Module):
    def __init__(self, in_dim=384, hidden=128, num_classes=2, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)
    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


head_full = WhisperHead(in_dim=X_full.shape[1])
print(f"Head params: {head_full.count_params():,}")

counts = np.bincount(y_full[train_idx_full].numpy(), minlength=2)
class_weights = torch.tensor(counts.sum() / (2.0 * np.maximum(counts, 1)), dtype=torch.float32)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(head_full.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

@torch.no_grad()
def evaluate_full(loader):
    head_full.eval()
    preds_all, labels_all = [], []
    for xb, yb in loader:
        preds_all.append(head_full(xb).argmax(-1).numpy())
        labels_all.append(yb.numpy())
    preds_all, labels_all = np.concatenate(preds_all), np.concatenate(labels_all)
    p, r, f1, _ = precision_recall_fscore_support(labels_all, preds_all, labels=[1], average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels_all, preds_all), "precision_eot": p, "recall_eot": r, "f1_eot": f1}

best_f1, patience = -1.0, 0
os.makedirs("checkpoints", exist_ok=True)

for epoch in range(30):
    head_full.train()
    running_loss = 0.0
    for xb, yb in train_loader_full:
        optimizer.zero_grad()
        loss = criterion(head_full(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    scheduler.step()
    vm = evaluate_full(val_loader_full)
    print(f"Epoch {epoch+1:02d} | loss={running_loss/len(train_idx_full):.4f} | "
          f"f1_eot={vm['f1_eot']:.4f} | precision={vm['precision_eot']:.4f} | recall={vm['recall_eot']:.4f}")
    if vm["f1_eot"] > best_f1:
        best_f1, patience = vm["f1_eot"], 0
        torch.save({"model_state": head_full.state_dict(), "in_dim": X_full.shape[1]}, "checkpoints/whisper_head_full_best.pt")
        print(f"  saved (f1={best_f1:.4f})")
    else:
        patience += 1
        if patience >= 6:
            print("Early stopping."); break

head_full.load_state_dict(torch.load("checkpoints/whisper_head_full_best.pt")["model_state"])
print("\nTest (full 5000-clip set):", evaluate_full(test_loader_full))

X: torch.Size([5000, 384]) y: torch.Size([5000]) balance: [2642, 2358]
Head params: 49,538
Epoch 01 | loss=0.6853 | f1_eot=0.6598 | precision=0.6389 | recall=0.6822
  saved (f1=0.6598)
Epoch 02 | loss=0.6466 | f1_eot=0.7241 | precision=0.6186 | recall=0.8729
  saved (f1=0.7241)
Epoch 03 | loss=0.6097 | f1_eot=0.7576 | precision=0.6573 | recall=0.8941
  saved (f1=0.7576)
Epoch 04 | loss=0.5834 | f1_eot=0.7295 | precision=0.6920 | recall=0.7712
Epoch 05 | loss=0.5617 | f1_eot=0.7127 | precision=0.7269 | recall=0.6992
Epoch 06 | loss=0.5494 | f1_eot=0.7738 | precision=0.6923 | recall=0.8771
  saved (f1=0.7738)
Epoch 07 | loss=0.5339 | f1_eot=0.7681 | precision=0.6966 | recall=0.8559
Epoch 08 | loss=0.5379 | f1_eot=0.7064 | precision=0.7700 | recall=0.6525
Epoch 09 | loss=0.5256 | f1_eot=0.7873 | precision=0.6727 | recall=0.9492
  saved (f1=0.7873)
Epoch 10 | loss=0.5045 | f1_eot=0.7218 | precision=0.7889 | recall=0.6653
Epoch 11 | loss=0.5027 | f1_eot=0.7692 | precision=0.7364 | recall=0.

In [ ]:

test_df_full = df_full.iloc[test_idx_full].reset_index(drop=True)
head_full.eval()
with torch.no_grad():
    test_preds_full = head_full(X_full[test_idx_full]).argmax(-1).numpy()
test_df_full["pred"] = test_preds_full
test_df_full["true_label"] = y_full[test_idx_full].numpy()

def slice_report_full(name, sub):
    if len(sub) == 0:
        print(f"{name}: no examples"); return
    p, r, f1, _ = precision_recall_fscore_support(sub["true_label"], sub["pred"], labels=[1], average="binary", zero_division=0)
    acc = accuracy_score(sub["true_label"], sub["pred"])
    print(f"{name:25s} n={len(sub):5d}  acc={acc:.3f}  f1_eot={f1:.3f}  precision={p:.3f}  recall={r:.3f}")

print("Stratified test results (Whisper Tiny + head, full 5000 clips):")
slice_report_full("overall", test_df_full)
slice_report_full("language=hin", test_df_full[test_df_full["language"] == "hin"])
slice_report_full("midfiller=True", test_df_full[test_df_full["midfiller"] == True])
slice_report_full("endfiller=True", test_df_full[test_df_full["endfiller"] == True])
slice_report_full("synthetic=True", test_df_full[test_df_full["synthetic"] == True])
slice_report_full("synthetic=False", test_df_full[test_df_full["synthetic"] == False])

Stratified test results (Whisper Tiny + head, full 5000 clips):
overall                   n=  500  acc=0.800  f1_eot=0.803  precision=0.750  recall=0.864
language=hin              n=  316  acc=0.775  f1_eot=0.805  precision=0.735  recall=0.891
midfiller=True            n=  299  acc=0.726  f1_eot=0.748  precision=0.682  recall=0.830
endfiller=True            n=  166  acc=0.861  f1_eot=0.000  precision=0.000  recall=0.000
synthetic=True            n=  478  acc=0.797  f1_eot=0.805  precision=0.749  recall=0.870
synthetic=False           n=   22  acc=0.864  f1_eot=0.727  precision=0.800  recall=0.667


In [8]:

from google.colab import files
files.download("/content/drive/MyDrive/Colab Notebooks/Task .ipynb")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>